# Cellpose Cell Segmentation & Fluorescence Analysis — Multi-Condition

Segments cells on a **reference channel** using Cellpose, applies masks to both
the reference and signal channels, and computes per-cell mean fluorescence and ratio.
Supports **any number of conditions**.

**Outputs:**
1. Bar graph — per-image mean `cellF_ratio` across all conditions, SEM, Mann-Whitney
2. Scatter plot — per-cell `cellF_sig` vs `cellF_ref` for all conditions overlaid
3. Violin plot — `cellF_ratio` distribution for all conditions, pairwise statistics


In [1]:
import os
import glob
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage, stats
from cellpose import models
from cellpose.utils import masks_to_outlines
import pandas as pd

print("All imports successful.")

All imports successful.


In [ ]:
# ─────────────────────────────────────────────
#  CONFIGURATION  (papermill injects CONDITIONS_CONFIG)
# ─────────────────────────────────────────────

# CONDITIONS_CONFIG: list of dicts with keys 'label' and 'root_path'
# Overridden by papermill / set interactively below
CONDITIONS_CONFIG = []

# Channel & Cellpose settings
REF_SUFFIX         = '_CH2.tif'
SIG_SUFFIX         = '_CH4.tif'
CELLPOSE_MODEL     = 'cyto3'
CELL_DIAMETER      = 0
FLOW_THRESHOLD     = 0.4
CELLPROB_THRESHOLD = 0.0
USE_GPU            = True


In [ ]:
# ─────────────────────────────────────────────
#  CONDITION SETUP
#  If CONDITIONS_CONFIG was not injected by papermill,
#  this cell prompts interactively for condition labels and paths.
# ─────────────────────────────────────────────
import os, re
import ipywidgets as widgets
from IPython.display import display

def _label_to_key(label, idx):
    key = re.sub(r'[^\w]', '_', label.strip()).lower()
    key = re.sub(r'_+', '_', key).strip('_')
    if not key or key[0].isdigit():
        key = 'cond_' + key if key else 'condition'
    return f'cond{idx}_{key}'

# ── Channel selection widgets ─────────────────────────────────────────────
CH_OPTIONS = ['CH1', 'CH2', 'CH3', 'CH4']

ref_toggle = widgets.ToggleButtons(
    options=CH_OPTIONS,
    value='CH2',
    description='Reference:',
    style={'description_width': '80px', 'button_width': '60px'},
    tooltips=['Segmentation channel'] * 4,
)
sig_toggle = widgets.ToggleButtons(
    options=CH_OPTIONS,
    value='CH4',
    description='Signal:',
    style={'description_width': '80px', 'button_width': '60px'},
    tooltips=['Fluorescence signal channel'] * 4,
)

print('Select channels (Cellpose segments on the Reference channel):')
display(ref_toggle, sig_toggle)

# ── Condition count & paths ───────────────────────────────────────────────
if not CONDITIONS_CONFIG:
    print('')
    n = int(input('How many conditions? '))
    CONDITIONS_CONFIG = []
    for i in range(n):
        label = input(f'  Label for condition {i+1}: ').strip()
        root  = input(f'  Root path for "{label}": ').strip()
        CONDITIONS_CONFIG.append({'label': label, 'key': _label_to_key(label, i), 'root_path': root})
else:
    for c in CONDITIONS_CONFIG:
        if 'key' not in c:
            c['key'] = _label_to_key(c['label'], CONDITIONS_CONFIG.index(c))

# ── Apply channel selections (only when running interactively) ─────────────
if not CONDITIONS_CONFIG:
    REF_SUFFIX = '_' + ref_toggle.value + '.tif'
    SIG_SUFFIX = '_' + sig_toggle.value + '.tif'

# Validate paths
for c in CONDITIONS_CONFIG:
    if not os.path.isdir(c['root_path']):
        raise ValueError(f"Path not found for condition '{c['label']}': {c['root_path']}")

# Derive SAVE_DIR
all_roots = [os.path.abspath(c['root_path']) for c in CONDITIONS_CONFIG]
SAVE_DIR  = os.path.join(os.path.commonpath(all_roots), 'RESULTS')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'\nReference channel : {REF_SUFFIX}')
print(f'Signal channel    : {SIG_SUFFIX}')
print(f'\n{len(CONDITIONS_CONFIG)} conditions configured:')
for c in CONDITIONS_CONFIG:
    print(f'  [{c["key"]}]  {c["label"]}  →  {c["root_path"]}')
print(f'Save dir: {SAVE_DIR}')


In [3]:
# ─────────────────────────────────────────────
#  HELPER FUNCTIONS
# ─────────────────────────────────────────────

def find_image_pairs(root_dir, ref_suffix, sig_suffix):
    """
    Recursively find all reference images under root_dir.
    Returns a list of (ref_path, sig_path) tuples where both files exist.
    """
    ref_files = sorted(glob.glob(os.path.join(root_dir, "**", f"*{ref_suffix}"), recursive=True))
    pairs = []
    for ref_path in ref_files:
        sig_path = ref_path.replace(ref_suffix, sig_suffix)
        if os.path.exists(sig_path):
            pairs.append((ref_path, sig_path))
        else:
            print(f"  [WARN] No matching signal file for: {ref_path}")
    return pairs


def load_channel_raw(path):
    """
    Load a .tif file and return a 2-D float32 image with ABSOLUTE pixel values.
    Original file is never modified.

    • Singleton dimensions are squeezed out.
    • RGB / multi-channel images are averaged to a single grayscale plane,
      preserving the original intensity scale.
    • No normalisation or rescaling is applied.
    """
    img = tifffile.imread(path).squeeze().astype(np.float32)

    if img.ndim == 3:
        if img.shape[0] <= 4:        # (C, H, W)
            img = img.mean(axis=0)
        elif img.shape[2] <= 4:      # (H, W, C)
            img = img.mean(axis=2)
        else:
            raise ValueError(
                f"Unexpected 3-D shape {img.shape} in {path}. "
                "Cannot determine channel axis."
            )
    elif img.ndim != 2:
        raise ValueError(f"Expected 2-D image after squeeze, got shape {img.shape} for {path}")

    return img   # float32, original intensity values — original file untouched


def to_8bit(img_raw):
    """
    Return a NEW uint8 copy of img_raw scaled 0-255. Used only for Cellpose input.
    The original img_raw array is never modified.
    """
    img = img_raw.copy()
    img_min, img_max = img.min(), img.max()
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min) * 255.0
    else:
        img = np.zeros_like(img)
    return img.astype(np.uint8)


def run_cellpose(model, img_8bit):
    """
    Run Cellpose on a 2-D uint8 reference image.
    Returns (masks, flows, styles).
    """
    masks, flows, styles = model.eval(
        img_8bit,
        diameter=CELL_DIAMETER,
        channels=[0, 0],
        flow_threshold=FLOW_THRESHOLD,
        cellprob_threshold=CELLPROB_THRESHOLD,
    )
    return masks, flows, styles


def save_seg_npy(ref_path, img_8bit, masks, flows, styles):
    """
    Save Cellpose segmentation output in the standard _seg.npy format.
    File is written next to the reference image as <basename>_seg.npy.
    """
    base = os.path.splitext(ref_path)[0]
    outlines = masks_to_outlines(masks)
    seg_dict = {
        "img"      : img_8bit,
        "masks"    : masks,
        "outlines" : outlines,
        "flows"    : flows,
        "styles"   : styles,
        "filename" : ref_path,
    }
    out_path = base + "_seg.npy"
    np.save(out_path, seg_dict)
    return out_path


def save_mask_overlay_png(ref_path, img_ref_raw, masks):
    """
    Save a PNG showing the reference channel image with coloured cell outlines
    overlaid. Saved next to the reference image as <basename>_overlay.png.
    Original image files are never modified.
    """
    base   = os.path.splitext(ref_path)[0]
    out_path = base + "_overlay.png"

    outlines = masks_to_outlines(masks)

    # Display image with percentile-based contrast (for visibility only)
    p_low, p_high = np.percentile(img_ref_raw, (1, 99))
    img_display = np.clip((img_ref_raw - p_low) / (p_high - p_low + 1e-6), 0, 1)

    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
    ax.imshow(img_display, cmap="gray", interpolation="nearest")

    # Colour-coded cell mask overlay (semi-transparent)
    if masks.max() > 0:
        mask_rgb = plt.cm.tab20(masks % 20)          # cycle 20 colours
        mask_rgb[masks == 0] = [0, 0, 0, 0]          # background transparent
        mask_rgb[..., 3] = np.where(masks > 0, 0.3, 0)  # 30% opacity fill
        ax.imshow(mask_rgb, interpolation="nearest")

        # White outlines on top
        outline_overlay = np.zeros((*outlines.shape, 4), dtype=np.float32)
        outline_overlay[outlines] = [1, 1, 1, 0.9]
        ax.imshow(outline_overlay, interpolation="nearest")

    n_cells = int(masks.max())
    ax.set_title(f"{os.path.basename(ref_path)}  |  {n_cells} cells", fontsize=9)
    ax.axis("off")
    plt.tight_layout(pad=0.5)
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


def extract_cell_fluorescence(mask, img_ref_raw, img_sig_raw):
    """
    For each labelled cell in mask, compute mean fluorescence using the
    ABSOLUTE (raw float32) pixel values of both channels.
    Returns arrays: cellF_ref, cellF_sig, cellF_ratio  (one value per cell).
    Cells with zero mean in ref are skipped to avoid division by zero.
    """
    cell_ids = np.unique(mask)
    cell_ids = cell_ids[cell_ids > 0]

    cellF_ref, cellF_sig, cellF_ratio = [], [], []
    for cid in cell_ids:
        px = mask == cid
        f_ref = img_ref_raw[px].mean()
        f_sig = img_sig_raw[px].mean()
        if f_ref == 0:
            continue
        cellF_ref.append(f_ref)
        cellF_sig.append(f_sig)
        cellF_ratio.append(f_sig / f_ref)

    return (np.array(cellF_ref, dtype=np.float32),
            np.array(cellF_sig, dtype=np.float32),
            np.array(cellF_ratio, dtype=np.float32))


def sig_stars(p):
    if p < 0.001:  return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else:          return "ns"


print("Helper functions defined.")

Helper functions defined.


In [ ]:
# ─────────────────────────────────────────────
#  MAIN PROCESSING LOOP
# ─────────────────────────────────────────────

cp_model = models.CellposeModel(model_type=CELLPOSE_MODEL, gpu=USE_GPU)
print(f"Cellpose model '{CELLPOSE_MODEL}' loaded (GPU={USE_GPU}).\n")

results        = {}   # keyed by condition key
per_cell_rows  = []
per_image_rows = []

for cond in CONDITIONS_CONFIG:
    label = cond['label']
    key   = cond['key']
    root  = cond['root_path']

    print(f"{'='*50}")
    print(f"Processing condition: {label}  |  {root}")

    pairs = find_image_pairs(root, REF_SUFFIX, SIG_SUFFIX)
    print(f"Found {len(pairs)} image pair(s).\n")

    img_means = []
    all_ref, all_sig, all_ratio = [], [], []

    for idx, (ref_path, sig_path) in enumerate(pairs):
        img_name = os.path.basename(ref_path)
        print(f"  [{idx+1}/{len(pairs)}] {img_name}", end=" ... ")

        img_ref_raw  = load_channel_raw(ref_path)
        img_sig_raw  = load_channel_raw(sig_path)
        img_ref_8bit = to_8bit(img_ref_raw)

        masks, flows, styles = run_cellpose(cp_model, img_ref_8bit)
        n_cells = int(masks.max())
        print(f"{n_cells} cells detected", end=" ... ")

        seg_path     = save_seg_npy(ref_path, img_ref_8bit, masks, flows, styles)
        overlay_path = save_mask_overlay_png(ref_path, img_ref_raw, masks)
        print("saved seg + overlay.")

        cellF_ref, cellF_sig, cellF_ratio = extract_cell_fluorescence(
            masks, img_ref_raw, img_sig_raw
        )
        n_valid = len(cellF_ratio)

        if n_valid > 0:
            img_means.append(float(cellF_ratio.mean()))
            all_ref.extend(cellF_ref.tolist())
            all_sig.extend(cellF_sig.tolist())
            all_ratio.extend(cellF_ratio.tolist())

            for i in range(n_valid):
                per_cell_rows.append({
                    'condition'    : label,
                    'image_file'   : img_name,
                    'image_path'   : ref_path,
                    'cell_id'      : i + 1,
                    'cellF_ref'    : float(cellF_ref[i]),
                    'cellF_sig'    : float(cellF_sig[i]),
                    'cellF_ratio'  : float(cellF_ratio[i]),
                })

            per_image_rows.append({
                'condition'          : label,
                'image_file'         : img_name,
                'image_path'         : ref_path,
                'n_cells_detected'   : n_cells,
                'n_cells_valid'      : n_valid,
                'mean_cellF_ref'     : float(cellF_ref.mean()),
                'mean_cellF_sig'     : float(cellF_sig.mean()),
                'mean_cellF_ratio'   : float(cellF_ratio.mean()),
                'sem_cellF_ratio'    : float(cellF_ratio.std(ddof=1) / np.sqrt(n_valid))
                                       if n_valid > 1 else float('nan'),
                'seg_npy_path'       : seg_path,
                'overlay_png_path'   : overlay_path,
            })

    results[key] = {
        'label'     : label,
        'img_means' : np.array(img_means),
        'all_ref'   : np.array(all_ref),
        'all_sig'   : np.array(all_sig),
        'all_ratio' : np.array(all_ratio),
    }
    print(f"\n  → {len(img_means)} images | {len(all_ratio)} total cells\n")

print('All processing complete.')


In [ ]:
# ─────────────────────────────────────────────
#  EXPORT TO EXCEL
# ─────────────────────────────────────────────

excel_path = os.path.join(SAVE_DIR, 'cellpose_results.xlsx')
df_cells  = pd.DataFrame(per_cell_rows)
df_images = pd.DataFrame(per_image_rows)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_cells.to_excel(writer,  sheet_name='per_cell',  index=False)
    df_images.to_excel(writer, sheet_name='per_image', index=False)

print('Excel saved: ' + excel_path)
print('  per_cell  sheet : ' + str(len(df_cells))  + ' rows')
print('  per_image sheet : ' + str(len(df_images)) + ' rows')
df_images.head()


In [ ]:
# ─────────────────────────────────────────────
#  FIGURE 1 — Bar graph: per-image mean cellF_ratio across all conditions
#  Error bars = SEM | pairwise Mann-Whitney U between all condition pairs
# ─────────────────────────────────────────────

from itertools import combinations
import matplotlib.cm as cm

keys   = [c['key']   for c in CONDITIONS_CONFIG]
labels = [c['label'] for c in CONDITIONS_CONFIG]
n_cond = len(keys)

# Colour palette: first two default to black/magenta, rest from tab10
default_colors = ['black', 'magenta']
tab_colors = [cm.tab10(i) for i in range(10)]
colors = [(default_colors[i] if i < len(default_colors) else tab_colors[i - len(default_colors)])
          for i in range(n_cond)]

means = [results[k]['img_means'] for k in keys]
bar_means = [m.mean() for m in means]
bar_sems  = [m.std(ddof=1)/np.sqrt(len(m)) if len(m)>1 else 0 for m in means]

fig, ax = plt.subplots(figsize=(1.2*n_cond + 1.5, 5))
positions = list(range(n_cond))

bar_kw = dict(width=0.5, edgecolor='black', linewidth=1.2, zorder=2)
for i, (pos, bm, bs, col) in enumerate(zip(positions, bar_means, bar_sems, colors)):
    ax.bar(pos, bm, color=col, **bar_kw)
    ax.errorbar(pos, bm, yerr=bs, fmt='none', color=col if col != 'black' else 'black',
                capsize=6, linewidth=1.5, zorder=3,
                ecolor='black')
    rng = np.random.default_rng(42+i)
    jitter = rng.uniform(-0.12, 0.12, size=len(means[i]))
    ax.scatter(pos + jitter, means[i], s=50, facecolors='none',
               edgecolors=col if col != 'black' else 'black',
               linewidth=1.2, zorder=4)

# Pairwise significance brackets
y_max = max(m.max() for m in means if len(m))
y_top = y_max * 1.12
h_step = y_max * 0.07
for level, (i, j) in enumerate(combinations(range(n_cond), 2)):
    stat, p = stats.mannwhitneyu(means[i], means[j], alternative='two-sided')
    stars = sig_stars(p)
    y = y_top + level * h_step
    h = h_step * 0.25
    ax.plot([i, i, j, j], [y, y+h, y+h, y], lw=1.0, color='black')
    fc = drug_mean = (bar_means[j] / bar_means[i]) if bar_means[i] != 0 else float('nan')
    ax.text((i+j)/2, y+h*1.2,
            str(round(fc, 2)) + 'x, ' + stars,
            ha='center', va='bottom', fontsize=8.5,
            color='black' if stars != 'ns' else 'gray')

ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Mean cellF_ratio (per image)', fontsize=11)
ax.set_title('cellF_ratio across conditions', fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'fig1_bar_cellF_ratio.svg'), format='svg', bbox_inches='tight')
plt.savefig(os.path.join(SAVE_DIR,'fig1_bar_cellF_ratio.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_bar_cellF_ratio')


In [ ]:
# ─────────────────────────────────────────────
#  FIGURE 2 — Scatter: cellF_sig vs cellF_ref, all conditions overlaid
# ─────────────────────────────────────────────

import matplotlib.cm as cm
default_colors = ['black', 'magenta']
tab_colors = [cm.tab10(i) for i in range(10)]
colors = [(default_colors[i] if i < len(default_colors) else tab_colors[i - len(default_colors)])
          for i in range(n_cond)]

fig, ax = plt.subplots(figsize=(5, 5))

for i, (key, col) in enumerate(zip(keys, colors)):
    r = results[key]
    ax.scatter(r['all_ref'], r['all_sig'], s=8, alpha=0.4, color=col, linewidths=0,
               label=r['label'] + ' (n=' + str(len(r['all_ref'])) + ' cells)')

ax.set_xlabel('cellF_ref (mean pixel intensity)', fontsize=11)
ax.set_ylabel('cellF_sig (mean pixel intensity)', fontsize=11)
ax.set_title('Per-cell fluorescence: signal vs reference', fontsize=11)
ax.legend(fontsize=8, frameon=False)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'fig2_scatter_sig_vs_ref.svg'), format='svg', bbox_inches='tight')
plt.savefig(os.path.join(SAVE_DIR,'fig2_scatter_sig_vs_ref.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig2_scatter_sig_vs_ref')


In [ ]:
# ─────────────────────────────────────────────
#  FIGURE 3 — Violin: cellF_ratio distribution, all conditions
#  Pairwise Mann-Whitney U between all condition pairs
# ─────────────────────────────────────────────

import matplotlib.cm as cm
default_colors = ['black', 'magenta']
tab_colors = [cm.tab10(i) for i in range(10)]
colors = [(default_colors[i] if i < len(default_colors) else tab_colors[i - len(default_colors)])
          for i in range(n_cond)]

ratios = [results[k]['all_ratio'] for k in keys]

fig, ax = plt.subplots(figsize=(1.2*n_cond + 1.5, 5))
positions = list(range(n_cond))

parts = ax.violinplot(ratios, positions=positions, showmedians=True,
                      showextrema=False, widths=0.6)
for body, col in zip(parts['bodies'], colors):
    body.set_facecolor(col)
    body.set_edgecolor(col)
    body.set_alpha(0.35)
parts['cmedians'].set_color('white')
parts['cmedians'].set_linewidth(2)

rng = np.random.default_rng(99)
for pos, arr, col in zip(positions, ratios, colors):
    jitter = rng.uniform(-0.12, 0.12, size=len(arr))
    ax.scatter(pos + jitter, arr, s=4, alpha=0.3, color=col, linewidths=0, zorder=3)

# Pairwise significance brackets
y_max = max(r.max() for r in ratios if len(r))
y_top = y_max * 1.08
h_step = y_max * 0.07
for level, (i, j) in enumerate(combinations(range(n_cond), 2)):
    stat, p = stats.mannwhitneyu(ratios[i], ratios[j], alternative='two-sided')
    stars = sig_stars(p)
    y = y_top + level * h_step
    h = h_step * 0.25
    ax.plot([i, i, j, j], [y, y+h, y+h, y], lw=1.0, color='black')
    ax.text((i+j)/2, y+h*1.2, stars, ha='center', va='bottom', fontsize=11,
            color='black' if stars != 'ns' else 'gray')

ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('cellF_ratio (sig / ref)', fontsize=11)
ax.set_title('cellF_ratio distribution across conditions', fontsize=10)
ax.spines[['top','right']].set_visible(False)

patches = [mpatches.Patch(color=c, alpha=0.6, label=l) for c, l in zip(colors, labels)]
ax.legend(handles=patches, fontsize=9, frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'fig3_violin_cellF_ratio.svg'), format='svg', bbox_inches='tight')
plt.savefig(os.path.join(SAVE_DIR,'fig3_violin_cellF_ratio.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_violin_cellF_ratio')
